# Part 7 — Advanced Topics

*Last updated:* 2026-11-12

This notebook collects advanced, production-oriented topics for power users:

- custom external database profiles (strict vs broad)
- programmatic YAML management (useful in pipelines)
- troubleshooting & diagnostics (common failure modes)
- restricted-server SSH bridging (ConnectionBridge)
- integration patterns (Snakemake/Nextflow-friendly workflows)

> **Tip:** You don’t need this notebook for day-to-day conversions. It’s here for when you want reproducibility at scale.


## 7.1 — Custom External Database Inclusion

The external YAML is an explicit **contract**: it defines which external namespaces are allowed to influence your graph.

Two common profiles:

- **Strict profile:** a small allowlist, lower ambiguity, smaller downloads.
- **Broad profile:** a larger allowlist (the shipped default is one), more coverage, but more ambiguity.

The YAML of a local repository has a fixed name, `<organism>_externals_modified.yml` (without it, the shipped default is used). A graph
file records the selection of external databases it was built from: loading it under another selection raises
`idtrack.GraphMismatchError` instead of answering from a graph of the other profile. So keep **one local repository per profile**: each
holds its YAML, its tables and its graph.

The next cell writes two profiles with `idtrack._default_externals.apply_allowlist`, which switches on exactly the
`{form: {database: [assembly keys]}}` entries of an allowlist in a template (the shipped templates list every database Ensembl has
for the organism, per form: gene, transcript, translation) and reports allowlisted names the template does not have. It writes
nothing over an existing YAML of your local repository.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import yaml

import idtrack
from idtrack._default_externals import DEFAULT_EXTERNALS, apply_allowlist
from idtrack._external_databases import write_externals_yaml

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)
print('Local repository:', LOCAL_REPOSITORY)

organism = 'homo_sapiens'
template_path = Path(idtrack.__file__).parent / 'default_config' / f'{organism}_externals_template.yml'
template = yaml.safe_load(template_path.read_text(encoding='utf-8'))
print('Template:', template_path.name)

# Allowlists name the databases of each form as the template does: {form: {database: [assembly keys]}}.
strict_allowlist = {
    'gene': {'HGNC Symbol': [36, 37, 38], 'EntrezGene': [36, 37, 38]},
    'transcript': {'RefSeq_mRNA': [36, 37, 38]},
    'translation': {'Uniprot/SWISSPROT': [36, 37, 38]},
}
broad_allowlist = DEFAULT_EXTERNALS[organism]  # the selection of the shipped default YAML

for name, allowlist in [('strict', strict_allowlist), ('broad', broad_allowlist)]:
    profile_repository = LOCAL_REPOSITORY.parent / f'idtrack_profile_{name}'  # one local repository per profile
    profile_repository.mkdir(parents=True, exist_ok=True)
    profile, enabled, absent = apply_allowlist(template, organism, allowlist)
    out_path = profile_repository / f'{organism}_externals_modified.yml'
    write_externals_yaml(profile, str(out_path))
    print(f'{name}: {enabled} form/database/assembly entries enabled; not in the template: {absent}')
    print('   written:', out_path)

print()
print('To use a profile: idtrack.API(local_repository=<its folder>), then build_graph as usual.')

Local repository: idtrack_cache
Template: homo_sapiens_externals_template.yml


strict: 10 form/database/assembly entries enabled; not in the template: []
   written: idtrack_profile_strict/homo_sapiens_externals_modified.yml


broad: 63 form/database/assembly entries enabled; not in the template: []
   written: idtrack_profile_broad/homo_sapiens_externals_modified.yml

To use a profile: idtrack.API(local_repository=<its folder>), then build_graph as usual.


## 7.2 — Programmatic YAML Management

For pipelines you often want YAML changes to be **scriptable** and **repeatable**.

A safe automation pattern:

1. Start from a template YAML (shipped in `idtrack/default_config/`, or generated as in Part 2).
2. Apply a curated allowlist in code (`apply_allowlist`).
3. Write the resulting YAML to a file you commit alongside your pipeline, and copy it into the local repository of the run.

The next cell shows a reusable helper that:
- reads a template YAML
- applies an allowlist
- writes a modified YAML to the path you give (never over a file you did not name)

> **Tip:** Keep allowlists per organism in your pipeline repository (as Python dicts or a small TOML/YAML). That makes your
> configuration reviewable. `idtrack._default_externals.DEFAULT_EXTERNALS` holds the allowlists of the shipped files.

In [2]:
from __future__ import annotations

import os
from pathlib import Path

import yaml

import idtrack
from idtrack._default_externals import apply_allowlist
from idtrack._external_databases import write_externals_yaml


def apply_allowlist_to_yaml(template_path: Path, organism: str, allowlist: dict, out_path: Path) -> None:
    template = yaml.safe_load(template_path.read_text(encoding='utf-8'))
    modified, enabled, absent = apply_allowlist(template, organism, allowlist)
    if absent:
        raise ValueError(f'Allowlisted databases that the template does not have: {absent}')
    write_externals_yaml(modified, str(out_path))
    print('Wrote:', out_path)
    print('Enabled form/database/assembly entries:', enabled)


# Example: a small mouse selection, written to a pipeline file (not into a local repository)
organism = 'mus_musculus'
template = Path(idtrack.__file__).parent / 'default_config' / f'{organism}_externals_template.yml'
out_yaml = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache')).parent / f'{organism}_externals_pipeline.yml'

allowlist_mouse = {
    'gene': {'MGI Symbol': [37, 38, 39], 'EntrezGene': [37, 38, 39]},
    'transcript': {'RefSeq_mRNA': [37, 38, 39]},
    'translation': {'Uniprot/SWISSPROT': [37, 38, 39]},
}
apply_allowlist_to_yaml(template, organism=organism, allowlist=allowlist_mouse, out_path=out_yaml)

Wrote: mus_musculus_externals_pipeline.yml
Enabled form/database/assembly entries: 12


## 7.3 — Troubleshooting & Diagnostics

Common failure modes (and what they usually mean):

- **Permission errors in `IDTRACK_LOCAL_REPO`:** your cache directory is not writable.
- **REST timeouts:** network issues or Ensembl REST is temporarily slow.
- **MySQL connection errors:** outbound MySQL ports may be blocked; IDTrack probes a port once and, when it fails, uses the HTTPS/FTP
  dumps for the next ten minutes or longer (the same tables, rows and values, slower per table). Set `IDTRACK_ENSEMBL_ACCESS=https` to
  skip MySQL entirely. Port `3337` is Ensembl's archive server (GRCh37 from release 79; IDTrack reads it for human GRCh37 only; its
  frozen copies of other assemblies carry later release numbers and are not used).
- **`idtrack.GraphMismatchError` when loading a graph:** the graph file in your local repository was built from another selection of
  external databases, another organism table or by an older graph format than you ask for now; the message names the differences.
  Move the file away (or use another local repository) and build again.
- **`idtrack.EnsemblDownloadError` / `idtrack.EnsemblRestError`:** Ensembl did not answer after the retries; the message names the URL.
  A release is never silently skipped because a request failed.
- **A warning that releases have no entry in the external YAML:** your external YAML does not include every release of the snapshot
  window; a graph built so has no external identifiers at those releases. Extend the file with `ExternalDatabases.extend_modified_yaml`
  (Part 2, section 2.6).
- **Unexpected 1→n explosion:** your external allowlist is too broad or contains promiscuous namespaces.

The next cell is a compact diagnostic report you can paste into issues or lab notes.


In [3]:
from __future__ import annotations

import os
import socket
from pathlib import Path

report = {}

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

report['local_repository'] = str(LOCAL_REPOSITORY)
report['local_repository_writable'] = os.access(LOCAL_REPOSITORY, os.W_OK)

# YAML + graph snapshot inventory
report['yaml_files'] = sorted(p.name for p in LOCAL_REPOSITORY.glob('*_externals_modified.yml'))
report['graph_snapshots'] = sorted(p.name for p in LOCAL_REPOSITORY.glob('graph_*.npz'))

# REST connectivity
try:
    import requests

    try:
        r = requests.get('https://rest.ensembl.org/info/ping', headers={'Content-Type': 'application/json'}, timeout=15)
        report['ensembl_rest_status'] = r.status_code
    except Exception as e:
        report['ensembl_rest_status'] = f'failed: {e.__class__.__name__}'
except Exception as e:
    report['ensembl_rest_status'] = f'requests_missing: {e.__class__.__name__}'

# MySQL ports (best-effort)
# Note: port 3337 is Ensembl's archive server; IDTrack reads it for human GRCh37 only. Other releases use 3306/5306.
# IDTrack itself probes a port once and remembers the answer (see `idtrack._ensembl_access`).
try:
    from idtrack._db import DB

    host = DB.mysql_host
    port_status = {}
    for port in [3306, 5306, 3337]:
        try:
            with socket.create_connection((host, port), timeout=2):
                port_status[port] = 'ok'
        except OSError as e:
            port_status[port] = e.__class__.__name__
    report['ensembl_mysql_ports'] = port_status
except Exception as e:
    report['ensembl_mysql_ports'] = f'skipped: {e.__class__.__name__}'

# Print report
for k, v in report.items():
    print(f'{k}: {v}')

# Optional: quick integrity check (only if the human graph of snapshot 115 exists)
# NOTE: TrackTests read Ensembl's tables too (a database manager); we only run one check here.
if any(g.startswith('graph_homo_sapiens_assembly38_min48_max115') for g in report['graph_snapshots']):
    try:
        import idtrack

        api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
        api.configure_logger()
        # Loads the graph of the pinned snapshot (it exists, see above).
        api.build_graph(organism_name='homo_sapiens', snapshot_release=115, return_test=True)

        check = api.track.is_edge_with_same_nts_only_at_backbone_nodes()  # a result: `holds`, `details`
        print()
        print('Quick TrackTests check (edge nts invariant):', check.holds)
    except Exception as e:
        print()
        print('TrackTests quick check skipped/failed ->', repr(e))


local_repository: idtrack_cache
local_repository_writable: True
yaml_files: ['homo_sapiens_externals_modified.yml', 'macaca_fascicularis_externals_modified.yml', 'macaca_mulatta_externals_modified.yml', 'mus_musculus_externals_modified.yml', 'sus_scrofa_externals_modified.yml']
graph_snapshots: ['graph_homo_sapiens_assembly38_min48_max115_narrow.npz', 'graph_macaca_fascicularis_assembly6_min91_max115_narrow.npz', 'graph_macaca_mulatta_assembly10_min48_max115_narrow.npz', 'graph_mus_musculus_assembly39_min48_max115_narrow.npz', 'graph_sus_scrofa_assembly111_min56_max115_narrow.npz']
ensembl_rest_status: 200
ensembl_mysql_ports: {3306: 'TimeoutError', 5306: 'TimeoutError', 3337: 'TimeoutError'}


2026-11-12 23:11:07 INFO:idtrack._database_manager: Ensembl releases of homo_sapiens assembly 38 read from 'idtrack_cache/ensembl_https_index_homo_sapiens_assembly-38.json' (scanned up to release 116).


2026-11-12 23:11:07 INFO:idtrack._graph_maker: The graph is being read: idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz



Quick TrackTests check (edge nts invariant): True


## 7.4 — Integration Patterns

IDTrack works best in pipelines when you make the snapshot boundary and cache location explicit.

Key ideas:

- Set `IDTRACK_LOCAL_REPO` to a stable, shared path (per project or per compute environment).
- Build snapshots once, then reuse them across jobs (a graph file loads without network and is only read, so parallel jobs can share
  one local repository).
- Pin the snapshot release in the pipeline: the graph file name, and the answers, depend on it.
- Store your external YAML alongside your pipeline so the configuration is reviewable.

Below are lightweight patterns you can adapt.


In [4]:
# Snakemake / Nextflow snippets (printed as plain text)

snakemake_rule = """
rule idtrack_build_human_graph:
    output:
        'idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz'
    shell:
        "python - <<'PY'\nimport os\nfrom pathlib import Path\nimport idtrack\n\nrepo = os.path.abspath('idtrack_cache')\nPath(repo).mkdir(parents=True, exist_ok=True)\n\napi = idtrack.API(local_repository=repo)\napi.build_graph(organism_name='homo_sapiens', snapshot_release=115)\nprint('Built: homo_sapiens 115')\nPY"
""".strip()

nextflow_process = """
process BUILD_IDTRACK_HUMAN {
  output:
    path "idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz"
  script:
    "export IDTRACK_LOCAL_REPO=$PWD/idtrack_cache\npython - <<'PY'\nimport os\nfrom pathlib import Path\nimport idtrack\n\nrepo = os.environ['IDTRACK_LOCAL_REPO']\nPath(repo).mkdir(parents=True, exist_ok=True)\n\napi = idtrack.API(local_repository=repo)\napi.build_graph(organism_name='homo_sapiens', snapshot_release=115)\nprint('Built: homo_sapiens 115')\nPY"
}
""".strip()

print('--- Snakemake example ---')
print(snakemake_rule)
print('--- Nextflow example ---')
print(nextflow_process)

--- Snakemake example ---
rule idtrack_build_human_graph:
    output:
        'idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz'
    shell:
        "python - <<'PY'
import os
from pathlib import Path
import idtrack

repo = os.path.abspath('idtrack_cache')
Path(repo).mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=repo)
api.build_graph(organism_name='homo_sapiens', snapshot_release=115)
print('Built: homo_sapiens 115')
PY"
--- Nextflow example ---
process BUILD_IDTRACK_HUMAN {
  output:
    path "idtrack_cache/graph_homo_sapiens_assembly38_min48_max115_narrow.npz"
  script:
    "export IDTRACK_LOCAL_REPO=$PWD/idtrack_cache
python - <<'PY'
import os
from pathlib import Path
import idtrack

repo = os.environ['IDTRACK_LOCAL_REPO']
Path(repo).mkdir(parents=True, exist_ok=True)

api = idtrack.API(local_repository=repo)
api.build_graph(organism_name='homo_sapiens', snapshot_release=115)
print('Built: homo_sapiens 115')
PY"
}


## 7.5 — No-Internet Servers: SSH ConnectionBridge

Some HPC clusters block outbound internet access from compute nodes. IDTrack needs to contact Ensembl services, so in
these environments you can route networking through your **local machine** using:

1) an SSH reverse SOCKS proxy (`ssh -R ...`) that creates a SOCKS5 proxy *on the remote machine*, and
2) `idtrack.ConnectionBridge` inside the Python process so Python uses that proxy.

### The most important rule

**The SOCKS proxy must exist on the same machine where your Python code is running.**

- If your Python runs on the login node → the proxy must be on the login node.
- If your Python runs on a compute node → the proxy must be on that compute node.
- If your Jupyter kernel runs on a compute node → the proxy must be on that compute node (not on the login node).

### Why “two SSH tabs” is normal (inbound vs outbound)

- **Inbound** traffic: your browser (local) needs to reach Jupyter (remote) → solved with `ssh -L` (Jupyter tunnel).
- **Outbound** traffic: your remote Python process needs to reach the internet (Ensembl) → solved with `ssh -R` +
  `idtrack.ConnectionBridge`.

These are independent problems, so using two separate SSH sessions/tabs is common. If your cluster allows you to SSH
directly to the machine running the kernel (often via `ssh -J login compute`), you can also combine `-L` and `-R`
in a single SSH command.

### Scenario 1 — Python runs on the server/login node (no compute node)

On your **local machine** (keep this SSH session open):

```bash
ssh -N -R 127.0.0.1:1080 user@server
```

On the **server** (run IDTrack inside this Python process):

```python
import idtrack

with idtrack.ConnectionBridge(proxy_port=1080):
    # ... run IDTrack code ...
    pass
```

### Scenario 2 — Python runs on a compute node (Slurm), not Jupyter (plain Python script)

This is the same idea, but the target machine is the **compute node**.

1) Start your job and learn the compute node hostname (often from job output or `hostname`).
2) On your **local machine**, create the proxy on that compute node (often via the login node as a jump host):

```bash
ssh -N -J user@login -R 127.0.0.1:1080 user@compute123
```

3) Run your Python script on the **compute node**, and enable `ConnectionBridge` at the start of the program.

### Scenario 3 — Jupyter kernel runs on a compute node (browser on your local machine)

You usually need BOTH:

- an **inbound** Jupyter tunnel (`ssh -L ...`) so your browser can reach Jupyter, and
- an **outbound** SOCKS tunnel (`ssh -R ...`) so the kernel can reach Ensembl.

Option A (common): keep your existing Jupyter tunnel workflow, and open a second SSH tab for the `ssh -R ...` tunnel.

Option B (single SSH session): only if you can SSH to the compute node (often via `-J`). Example (adjust ports/users):

```bash
ssh -N -J user@login -L 8888:127.0.0.1:8888 -R 127.0.0.1:1080 user@compute123
```

### What `ConnectionBridge` changes (briefly)

- It patches `socket.socket` to PySocks’ `socks.socksocket` so most Python networking stacks transparently use the proxy
  (requests/urllib3, urllib, PyMySQL, etc.).
- It optionally sets `ALL_PROXY`/`all_proxy` so child processes inherit the proxy.
- It restores the original state on `stop()` (also best-effort on interpreter exit), including a default proxy that was set before.
- A bridge stays active until `stop()` (or `ConnectionBridge.stop_all()`, which ends it whoever started it); dropping the object does
  not end it.

> **Tip:** Prefer the context-manager form (`with ConnectionBridge(...)`) in scripts so cleanup is guaranteed. It raises
> `ConnectionError` when the connectivity check fails, so the block never runs without the network it needs.


### Copy/paste helper: ConnectionBridge SSH command

There is a ready-to-use helper script in this repo:

- `idtrack/reproducibility/scripts/connection_bridge_tunnel.sh`

It is meant to run on your **local machine**. It prints only the SSH command (one line). Paste that line into a
terminal tab and keep it open.

Examples:

```bash
# Show full beginner-oriented help:
bash idtrack/reproducibility/scripts/connection_bridge_tunnel.sh --help

# Server/login node (Python runs on the login node):
bash idtrack/reproducibility/scripts/connection_bridge_tunnel.sh user@server

# Compute node (Python runs on compute node; go via login as jump host):
bash idtrack/reproducibility/scripts/connection_bridge_tunnel.sh --jump user@login compute123

# Compute node host auto-detected from Jupyter URL (kernel on compute node):
bash idtrack/reproducibility/scripts/connection_bridge_tunnel.sh --jump user@login "http://compute123:8888/lab?token=..."

# Optional: one combined SSH command with both Jupyter (-L) and SOCKS (-R):
bash idtrack/reproducibility/scripts/connection_bridge_tunnel.sh --jump user@login --with-jupyter "http://compute123:8888/lab?token=..."
```


In [5]:
from __future__ import annotations

import idtrack

# Enable the bridge inside the current Python process (e.g. a Jupyter kernel running on the server).
# If `test=True` (default), the bridge pings Ensembl REST and rolls back automatically on failure.
b = idtrack.ConnectionBridge(proxy_port=1080)
ok = b.start(test=True)
print('Bridge enabled:', ok)

# ... run IDTrack code here ...
# api = idtrack.API(local_repository='./idtrack_cache')
# api.build_graph(organism_name='homo_sapiens', snapshot_release=115)

# Recommended pattern in scripts:
# with idtrack.ConnectionBridge(proxy_port=1080) as _b:
#     ... do work ...

b.stop()
print('Bridge disabled')


2026-11-12 23:11:45 INFO:idtrack._connection_bridge: [idtrack] ConnectionBridge enabled: all TCP sockets in this Python process are routed through 127.0.0.1:1080. Call `b.stop()` to restore normal networking.


2026-11-12 23:11:45 ERROR:idtrack._connection_bridge: [idtrack] ConnectionBridge check failed: SOCKSHTTPSConnectionPool(host='rest.ensembl.org', port=443): Max retries exceeded with url: /info/ping (Caused by NewConnectionError("SOCKSHTTPSConnection(host='rest.ensembl.org', port=443): Failed to establish a new connection: [Errno 111] Connection refused"))
  Troubleshooting:
  - Ensure the SSH session is established from your local machine: `ssh -R 1080 user@server`
  - On the server, check the SOCKS port is listening: `ss -tlnp | grep 1080`


2026-11-12 23:11:45 WARNING:idtrack._connection_bridge: [idtrack] ConnectionBridge test failed; disabling bridge.


2026-11-12 23:11:45 INFO:idtrack._connection_bridge: [idtrack] ConnectionBridge disabled: normal networking restored.


2026-11-12 23:11:45 INFO:idtrack._connection_bridge: [idtrack] ConnectionBridge already stopped.


[idtrack] ConnectionBridge enabled: all TCP sockets in this Python process are routed through 127.0.0.1:1080. Call `b.stop()` to restore normal networking.
[idtrack] ConnectionBridge check failed: SOCKSHTTPSConnectionPool(host='rest.ensembl.org', port=443): Max retries exceeded with url: /info/ping (Caused by NewConnectionError("SOCKSHTTPSConnection(host='rest.ensembl.org', port=443): Failed to establish a new connection: [Errno 111] Connection refused"))
  Troubleshooting:
  - Ensure the SSH session is established from your local machine: `ssh -R 1080 user@server`
  - On the server, check the SOCKS port is listening: `ss -tlnp | grep 1080`
[idtrack] ConnectionBridge test failed; disabling bridge.
[idtrack] ConnectionBridge disabled: normal networking restored.
Bridge enabled: False
[idtrack] ConnectionBridge already stopped.
Bridge disabled
